# 72 — Build LGBM features + train LambdaRank (Stage C)

Walks HF train conversations, splits sessions 80/20, runs the full
Stage A+B retrieval+reranker pipeline to get top-100 candidates per
music turn, then extracts the extended 28-feature vectors per
(turn, candidate) pair. Trains LightGBM LambdaRank on the result.

**Prereqs**: Stage A + Stage B done; merged BGE-M3 + CE on Hub;
BGE-M3-FT catalog pickle on Drive (notebook 70 cell 6).

**Wallclock**: ~4-6 hr on Blackwell (feature extraction is wRRF +
CE forward over ~12k music turns × 100 cands).

In [ ]:
# 1) Setup.
import os
from google.colab import userdata, drive
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
drive.mount('/content/drive', force_remount=False)

BRANCH = 'fresh-model'
!rm -rf /content/recsys2026
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026
%cd /content/recsys2026

DRIVE_BASE = '/content/drive/MyDrive'
LOCAL_BASE = '/content/recsys2026/experiments/cache'
os.makedirs(LOCAL_BASE, exist_ok=True)
src = f'{DRIVE_BASE}/recsys2026_retrieval_v2_cache'
dst = f'{LOCAL_BASE}/retrieval_v2'
if os.path.islink(dst): os.unlink(dst)
elif os.path.exists(dst):
    import shutil; shutil.rmtree(dst)
os.symlink(src, dst)

!pip install -q --upgrade lightgbm sentence-transformers transformers datasets scikit-learn

In [ ]:
# 2) Walk HF train conversations + session-disjoint 80/20 split.
import sys
sys.path.insert(0, '/content/recsys2026/scripts')
sys.path.insert(0, '/content/recsys2026/music-crs-baselines')
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from build_bi_encoder_training_data import _iter_conversation_turns

train_conv = load_dataset('talkpl-ai/TalkPlayData-Challenge-Dataset', split='train')
all_rows = _iter_conversation_turns(train_conv)
print(f'{len(all_rows)} per-music-turn rows from train split')
session_ids = sorted({r['session_id'] for r in all_rows})
train_sids, val_sids = train_test_split(session_ids, test_size=0.2, random_state=42)
train_set, val_set = set(train_sids), set(val_sids)
train_rows = [r for r in all_rows if r['session_id'] in train_set]
val_rows = [r for r in all_rows if r['session_id'] in val_set]
print(f'train turns: {len(train_rows)}  val turns: {len(val_rows)}')
import json as _j
os.makedirs('experiments/cache/retrieval_v2/lgbm', exist_ok=True)
with open('experiments/cache/retrieval_v2/lgbm/lgbm_train_rows.jsonl', 'w') as f:
    for r in train_rows: f.write(_j.dumps(r, default=str) + '\n')
with open('experiments/cache/retrieval_v2/lgbm/lgbm_val_rows.jsonl', 'w') as f:
    for r in val_rows: f.write(_j.dumps(r, default=str) + '\n')

In [ ]:
# 3) Extract features for each (turn, candidate) pair via Stage A+B pipeline.
# For each music turn:
#   a. Build production query via format_query_text(..., mode='raw').
#   b. wRRF (BM25 + dense_lyrics + BGE-M3-FT) → top-100 candidates.
#   c. CE rerank → top-100 reranked (keeps the same 100 candidates; just adds CE score).
#   d. extract_features() with the extended 28-feature vector.
# Outputs: experiments/cache/retrieval_v2/lgbm/lgbm_{train,val}_features.parquet
!cd /content/recsys2026/music-crs-baselines && python -u ../scripts/build_lgbm_features.py \
    --n-sessions 999999 \
    --topk 100 \
    --seed 42 \
    --out /content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_train_features.parquet \
    --cache-dir /content/recsys2026/experiments/cache \
    2>&1 | tail -20
# Note: the existing build_lgbm_features.py samples train sessions; with the
# new 80/20 split, override the sampler by writing a thin per-row driver.
# Implementation detail: pass session_ids filter via the existing --seed +
# n-sessions, OR modify build_lgbm_features.py to accept --session-id-list.
# For Phase 1, the simpler path is: run the full feature extractor on train,
# then post-filter rows to (train_sids, val_sids) into two parquets.

In [ ]:
# 4) Post-filter the single full-train parquet into 80/20 train/val by session.
import pandas as pd
df = pd.read_parquet('/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_train_features.parquet')
tdf = df[df['session_id'].isin(train_set)].copy()
vdf = df[df['session_id'].isin(val_set)].copy()
tdf.to_parquet('/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_train_split.parquet', index=False)
vdf.to_parquet('/content/recsys2026/experiments/cache/retrieval_v2/lgbm/lgbm_val_split.parquet', index=False)
print(f'train rows: {len(tdf)}  val rows: {len(vdf)}')
print('positives (label=1):', int(tdf['label'].sum()), int(vdf['label'].sum()))